# Repeated walk-forward strategy evaluation

A single historical test period can be unrepresentative. This notebook repeats the full train/validation/test procedure across several non-overlapping test periods. Each fold selects a strategy using validation data only, then evaluates that selected strategy on the following untouched test interval.

In [ ]:
import numpy as np
import pandas as pd

from cobasket.repeated_walk_forward import WalkForwardConfig, run_repeated_walk_forward
from cobasket.strategy_experiments import StrategyExperimentConfig
from cobasket.strategy_rules import MetricCondition, StrategyRule, StrategyRules

## Synthetic prices and a probability metric

The example is deliberately synthetic so it runs without network access. Replace these tables with the historical prices and leakage-safe metrics generated by Cobasket.

In [ ]:
index = pd.date_range('2018-01-01', periods=900, freq='B')
x = np.arange(len(index), dtype=float)
prices = pd.DataFrame({
    'AAA': 100 * np.exp(0.0007 * x + 0.03 * np.sin(x / 35)),
    'BBB': 100 * np.exp(0.0004 * x + 0.02 * np.cos(x / 45)),
}, index=index)
probability = pd.DataFrame({
    'AAA': 0.60 + 0.15 * np.sin(x / 40),
    'BBB': 0.60 + 0.15 * np.cos(x / 40),
}, index=index).clip(0, 1)
metrics = {'probability': probability}

In [ ]:
conservative = StrategyRules(
    name='conservative',
    rules=(StrategyRule('buy', (MetricCondition('probability', '>=', 0.70),), 0.20),),
)
permissive = StrategyRules(
    name='permissive',
    rules=(StrategyRule('buy', (MetricCondition('probability', '>=', 0.58),), 0.20),),
)
strategies = (conservative, permissive)

In [ ]:
result = run_repeated_walk_forward(
    prices,
    metrics,
    strategies,
    walk_forward=WalkForwardConfig(
        train_observations=300,
        validation_observations=100,
        test_observations=100,
        step_observations=100,
    ),
    experiment=StrategyExperimentConfig(
        selection_metric='sharpe_ratio',
        minimum_train_observations=50,
        minimum_validation_observations=50,
        minimum_test_observations=50,
    ),
)

In [ ]:
result.fold_table

In [ ]:
result.selection_frequency

In [ ]:
result.compounded_equity.plot(title='Compounded non-overlapping test-fold returns')

The compounded curve is an out-of-sample summary, not a literal continuous portfolio simulation: each fold starts from the configured cash level and does not inherit positions from the previous fold. The selection-frequency table is equally important. If the preferred strategy changes frequently, the decision rules are regime-dependent.